# Experiment 008: warm-start MC-SFT, the last prep x method cell (Colab)

007 showed mc-sft from the raw base matches grpo at 3b and beats it at 0.5b.
This run fills the remaining combination: gsm8k-sft first (the exact stage-1
adapters that fed grpo in experiments 006 and 003), then mc-sft on the same 245
mmlu prompts. Head-to-head against 006/003 grpo this isolates what the second
stage contributes from an identical warm start. Exploratory, the registered
pipeline stays untouched.

Scope: 8 passes (123 steps) at both scales. I almost used 2 passes at 3b (the
format saturation point) but the hard tier kept improving to 8 passes in 007
and hard is the tier this experiment is about, so both scales run 8, which also
gives a matched-exposure comparison against the 007 p8 checkpoints. 3 seeds
probe-only, then generative evals for the seed-0 checkpoints. Expectation
stated up front: ceiling-level equivalence on easy/medium, the informative
tier is hard, where grpo held its only edge at 3b. ~3h on an a100.

## 0. Compute units and mount Drive

Fill `cu_before` now. The open record is saved straight after Drive mounts,
so a disconnected session can be closed when the next runtime starts.

In [4]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 485.12   # e.g. 87.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close an unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '008_colab_ws_mc_sft', 'colab_cu_before': cu_before}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260718_113014 | cu_before = 485.12
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive mounted -> /content/drive/MyDrive/inm363


## 1. Check the GPU


In [5]:
!nvidia-smi -L
import torch
print('cuda:', torch.cuda.is_available(), '| torch:', torch.__version__)


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-343c51c5-83c9-a00f-01c2-106b23b1165c)
cuda: True | torch: 2.11.0+cu128


## 2. Clone the repo (automatic)

This cell clones the main branch of the public repository without asking for a GitHub token. For online experiment tracking, add `WANDB_API_KEY` under the key icon in Colab Secrets and grant this notebook access. If using a private fork, set `REPO_SLUG` to its owner and name, then add a read-only `GH_TOKEN` for that fork. A missing token for a private fork triggers a one-time prompt.


In [6]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 1203, done.
remote: Counting objects: 100% (460/460), done.
remote: Compressing objects: 100% (296/296), done.
remote: Total 1203 (delta 239), reused 376 (delta 161), pack-reused 743 (from 1)
Receiving objects: 100% (1203/1203), 30.94 MiB | 40.11 MiB/s, done.
Resolving deltas: 100% (658/658), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

Keep colabs own cuda torch, just pin the libraries that drifted.


In [7]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 136.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 54.8 MB/s eta 0:00:00
deps installed


## 4. W&B key (automatic)

Reads `WANDB_API_KEY` from Colab Secrets and writes it to the file the code expects.
If the secret isn't set, it falls back to a manual upload of `wandb_api_key.txt`.


In [8]:
import os

# 1) Use the key file when it is already present.
# 2) Else read WANDB_API_KEY from colab secrets and write the file.
# 3) Else fall back to a manual upload.
if os.path.exists('wandb_api_key.txt'):
    print('wandb_api_key.txt already present, using it')
else:
    wkey = None
    try:
        from google.colab import userdata
        wkey = userdata.get('WANDB_API_KEY')
    except Exception:
        wkey = None
    if wkey:
        open('wandb_api_key.txt', 'w').write(wkey.strip())
        print('wrote wandb key from colab secret')
    else:
        from google.colab import files
        files.upload()  # pick wandb_api_key.txt
print('have key file:', os.path.exists('wandb_api_key.txt'))


wrote wandb key from colab secret
have key file: True


## 5. Check the Drive record

Outputs (results + checkpoints) are written straight to drive via `output_root`
in the config, so they survive a colab disconnect and a crashed run can resume.


In [9]:
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')


open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260718_113014.json


## 6. Build the data splits

Materialises the mmlu splits and runs the leakage check. The same 245 deduped
dev+val prompts feed the mc-sft targets here instead of grpo rollouts.


In [10]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 17.1MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:00<00:00, 4.77MB/s,  229kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:00<00:00, 4.78MB/s,  229kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:  63% 264k/419k [00:00<00:00, 673kB/s,   ???B/s  ]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:00<00:00, 801kB/s, 41.5kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:00<00:00, 800kB/s, 41.5kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 375877.32 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 296589.66 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 82.1MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 156MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00

## 7. Preflight: the stage-1 adapters must be on Drive

The whole experiment chains from the gsm8k-sft checkpoints of 006 (0.5b) and
003 (3b). If either is missing the run would die an hour in, so check first.

In [11]:
import os
D = '/content/drive/MyDrive/inm363'
stage1 = ['checkpoints/006_COLAB_FULL_B05_sft_grpo_seed0/sft',
          'checkpoints/003_COLAB_FULL_B3_sft_grpo_seed0/sft']
for rel in stage1:
    # the weights for the merge, and the provenance file the base resolver needs
    for fname in ('adapter_model.safetensors', 'adapter_base.json'):
        p = os.path.join(D, rel, fname)
        assert os.path.exists(p), f'stage-1 file missing: {p}'
    print('found', rel)
print('both warm starts present and complete')

found checkpoints/006_COLAB_FULL_B05_sft_grpo_seed0/sft
found checkpoints/003_COLAB_FULL_B3_sft_grpo_seed0/sft
both warm starts present and complete


## 8. Wiring check

In [12]:
!python scripts/run.py --config configs/008_wsft_colab_b05.yaml

env=COLAB mode=FULL scale=B05 condition=SFT_ONLY
model=Qwen/Qwen2.5-0.5B device=cuda stages=['base_eval', 'sft', 'sft_eval']
steps=123 N=8 seed=0
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/008_wsft_COLAB_FULL_B05_sft_only_seed0/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/008_wsft_COLAB_FULL_B05_sft_only_seed0/
local smoke wiring ok


## 9. The training sweep (probe-only)

6 runs: 8 passes per scale x 3 seeds. The stage-1 adapter is fixed (seed 0
artifact), the seed varies only the mc-sft stage, same shape as 007. The first
run per scale does the one-off merge of the stage-1 adapter into the base,
seeds 1 and 2 reuse the shared merged model.

**If colab disconnects:** re-run the cell, finished runs skip in seconds.

In [13]:
# python loop with ! calls so the logs stream live, a bash cell would buffer
# everything until the end and look frozen
sweep = ['008_wsft_colab_b05', '008_wsft_colab_b3']
for c in sweep:
    for s in (0, 1, 2):
        print(f'===== {c} seed {s} =====')
        !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/{c}.yaml --seed {s} --stage all

===== 008_wsft_colab_b05 seed 0 =====
seed overridden to 0
2026-07-18 11:33:43,249 INFO loaded wandb api key from file
2026-07-18 11:33:43,250 INFO run 008_wsft_COLAB_FULL_B05_sft_only_seed0 device=cuda stages=['base_eval', 'sft', 'sft_eval'] trackers=['tensorboard', 'wandb'] out=/content/drive/MyDrive/inm363/results/008_wsft_COLAB_FULL_B05_sft_only_seed0
2026-07-18 11:33:43,250 INFO base_eval skipped, probe-only run
config.json: 100% 681/681 [00:00<00:00, 4.66MB/s]
tokenizer_config.json: 100% 7.23k/7.23k [00:00<00:00, 27.6MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 41.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 112MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 130MB/s]

model.safetensors: downloading bytes:  10% 99.0M/988M [00:01<00:06, 142MB/s, 7.85MB/s  ]
model.safetensors: reconstructing file:  11% 112M/988M [00:01<00:10, 81.8MB/s]
model.safetensors: reconstructing file:  13% 126M/988M [00:01<00:13, 63.2MB/s, 10.9MB/s  ]
model.safetensors: downloading bytes:  12% 1

## 10. Generative evals (seed 0, both scales)

In [14]:
# same live-streaming loop style as the sweep above
D = '/content/drive/MyDrive/inm363'
for suffix, sc in [('b05', 'B05'), ('b3', 'B3')]:
    tag = f'008_wsft_COLAB_FULL_{sc}_sft_only_seed0'
    print(f'===== gen eval {tag} =====')
    !python scripts/eval_adapter.py --config configs/008_wsft_colab_{suffix}.yaml --adapter {D}/checkpoints/{tag}/sft --condition sft --out {D}/results/{tag}

===== gen eval 008_wsft_COLAB_FULL_B05_sft_only_seed0 =====
Loading weights: 100% 290/290 [00:00<00:00, 2513.25it/s]
sft eval (1884 items): 100% 118/118 [04:40<00:00,  2.37s/it, acc 0.535 fmt 0.975]
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.6621160409556314, 'medium': 0.4729064039408867, 'hard': 0.34595959595959597}
probe per-tier: {'easy': 0.6860068259385665, 'medium': 0.5123152709359606, 'hard': 0.31565656565656564}
===== gen eval 008_wsft_COLAB_FULL_B3_sft_only_seed0 =====
Loading weights: 100% 434/434 [00:00<00:00, 2988.95it/s]
sft eval (1884 items): 100% 118/118 [05:31<00:00,  2.81s/it, acc 0.732 fmt 1.000]
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.8430034129692833, 'medium': 0.7438423645320197, 'hard': 0.4696969696969697}
probe per-tier: {'easy': 0.8532423208191127, 'medium': 0.7405582922824302, 'hard': 0.5202020202020202}


## 11. Peek at the results

Probe accuracies for all 6 runs, generative where already evaluated.

In [15]:
import json, os
D = '/content/drive/MyDrive/inm363'
for sc in ('B05', 'B3'):
    for s in (0, 1, 2):
        tag = f'008_wsft_COLAB_FULL_{sc}_sft_only_seed{s}'
        d = os.path.join(D, 'results', tag)
        lp_f = os.path.join(d, 'sft_logprob.json')
        ev_f = os.path.join(d, 'sft_eval.json')
        lp = json.load(open(lp_f))['per_tier_acc'] if os.path.exists(lp_f) else None
        ev = json.load(open(ev_f))['per_tier_acc'] if os.path.exists(ev_f) else None
        fmt = lambda x: {k: round(v, 3) for k, v in x.items()} if x else '-'
        print(f'{tag}  probe {fmt(lp)}  gen {fmt(ev)}')

008_wsft_COLAB_FULL_B05_sft_only_seed0  probe {'easy': 0.686, 'medium': 0.512, 'hard': 0.316}  gen {'easy': 0.662, 'medium': 0.473, 'hard': 0.346}
008_wsft_COLAB_FULL_B05_sft_only_seed1  probe {'easy': 0.688, 'medium': 0.509, 'hard': 0.338}  gen -
008_wsft_COLAB_FULL_B05_sft_only_seed2  probe {'easy': 0.686, 'medium': 0.516, 'hard': 0.343}  gen -
008_wsft_COLAB_FULL_B3_sft_only_seed0  probe {'easy': 0.853, 'medium': 0.741, 'hard': 0.52}  gen {'easy': 0.843, 'medium': 0.744, 'hard': 0.47}
008_wsft_COLAB_FULL_B3_sft_only_seed1  probe {'easy': 0.858, 'medium': 0.744, 'hard': 0.528}  gen -
008_wsft_COLAB_FULL_B3_sft_only_seed2  probe {'easy': 0.854, 'medium': 0.744, 'hard': 0.52}  gen -


## 12. Close the units record

In [16]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 475.36   # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
